In [52]:
import pandas as pd
import numpy as np
import json as js
import re

In [53]:
# Load the data set
file_path= "../data/raw/scout_car.json"
df = pd.read_json(file_path, lines=True)
df.head()

,url,make_model,short_description,body_type,price,vat,km,registration,prev_owner,kW,...,description,Emission Label,Gears,Country version,Electricity consumption,Last Service Date,Other Fuel Types,Availability,Last Timing Belt Service Date,Available from
0,https://www.autoscout24.com//offers/audi-a1-sp...,Audi A1,Sportback 1.4 TDI S-tronic Xenon Navi Klima,Sedans,15770,VAT deductible,"56,013 km",01/2016,2 previous owners,NaN,...,"[\n, Sicherheit:, , Deaktivierung für Beifahr...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://www.autoscout24.com//offers/audi-a1-1-...,Audi A1,1.8 TFSI sport,Sedans,14500,Price negotiable,"80,000 km",03/2017,None,NaN,...,[\nLangstreckenfahrzeug daher die hohe Kilomet...,[\n4 (Green)\n],[\n7\n],NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://www.autoscout24.com//offers/audi-a1-sp...,Audi A1,Sportback 1.6 TDI S tronic Einparkhilfe plus+m...,Sedans,14640,VAT deductible,"83,450 km",02/2016,1 previous owner,NaN,...,"[\n, Fahrzeug-Nummer: AM-95365, , Ehem. UPE 2...",[\n4 (Green)\n],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://www.autoscout24.com//offers/audi-a1-1-...,Audi A1,1.4 TDi Design S tronic,Sedans,14500,None,"73,000 km",08/2016,1 previous owner,NaN,...,"[\nAudi A1: , - 1e eigenaar , - Perfecte staat...",NaN,[\n6\n],NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://www.autoscout24.com//offers/audi-a1-sp...,Audi A1,Sportback 1.4 TDI S-Tronic S-Line Ext. admired...,Sedans,16790,None,"16,200 km",05/2016,1 previous owner,NaN,...,"[\n, Technik & Sicherheit:, Xenon plus, Klimaa...",NaN,NaN,[\nGermany\n],NaN,NaN,NaN,NaN,NaN,NaN


In [54]:
# Check the data 
df.shape
#df.info()
#df.columns

(15919, 54)

In [55]:
# Data Cleanning

In [56]:
# The data set columns are problematic- they contain spaces, special characters(&, .), 
# hidden newline characters(\n).
# They are also inconsistent(some lowercase, some uppercase)

In [57]:
# Clean and make consistent column names
df.columns = (df.columns
              .str.strip() # Remove extra spaces
              .str.lower() # Convert everything to lowercase
              .str.replace("\n", "", regex=False) # Remove hidden newline characters
              .str.replace("&","and", regex = False) # Replace "&" with "and"
              .str.replace(".","", regex = False) # Remove periods
              .str.replace(" ","_") # Replace spaces with underscores


)

In [58]:
# Drop broken / useless columns
drop_cols = [
    "kw",
    "url",
    "offer_number",
    "model_code",
    "electricity_consumption",
    "last_service_date",
    "other_fuel_types",
    "availability",
    "last_timing_belt_service_date",
    "available_from",
    "null"
]

df = df.drop(columns=drop_cols, errors="ignore")

In [59]:
df.head()

,make_model,short_description,body_type,price,vat,km,registration,prev_owner,hp,type,...,co2_emission,emission_class,comfort_and_convenience,entertainment_and_media,extras,safety_and_security,description,emission_label,gears,country_version
0,Audi A1,Sportback 1.4 TDI S-tronic Xenon Navi Klima,Sedans,15770,VAT deductible,"56,013 km",01/2016,2 previous owners,66 kW,"[, Used, , Diesel (Particulate Filter)]",...,[\n99 g CO2/km (comb)\n],[\nEuro 6\n],"[Air conditioning, Armrest, Automatic climate ...","[Bluetooth, Hands-free equipment, On-board com...","[Alloy wheels, Catalytic Converter, Voice Cont...","[ABS, Central door lock, Daytime running light...","[\n, Sicherheit:, , Deaktivierung für Beifahr...",NaN,NaN,NaN
1,Audi A1,1.8 TFSI sport,Sedans,14500,Price negotiable,"80,000 km",03/2017,None,141 kW,"[, Used, , Gasoline]",...,[\n129 g CO2/km (comb)\n],[\nEuro 6\n],"[Air conditioning, Automatic climate control, ...","[Bluetooth, Hands-free equipment, On-board com...","[Alloy wheels, Sport seats, Sport suspension, ...","[ABS, Central door lock, Central door lock wit...",[\nLangstreckenfahrzeug daher die hohe Kilomet...,[\n4 (Green)\n],[\n7\n],NaN
2,Audi A1,Sportback 1.6 TDI S tronic Einparkhilfe plus+m...,Sedans,14640,VAT deductible,"83,450 km",02/2016,1 previous owner,85 kW,"[, Used, , Diesel (Particulate Filter)]",...,[\n99 g CO2/km (comb)\n],[\nEuro 6\n],"[Air conditioning, Cruise control, Electrical ...","[MP3, On-board computer]","[Alloy wheels, Voice Control]","[ABS, Central door lock, Daytime running light...","[\n, Fahrzeug-Nummer: AM-95365, , Ehem. UPE 2...",[\n4 (Green)\n],NaN,NaN
3,Audi A1,1.4 TDi Design S tronic,Sedans,14500,None,"73,000 km",08/2016,1 previous owner,66 kW,"[, Used, , Diesel (Particulate Filter)]",...,[\n99 g CO2/km (comb)\n],[\nEuro 6\n],"[Air suspension, Armrest, Auxiliary heating, E...","[Bluetooth, CD player, Hands-free equipment, M...","[Alloy wheels, Sport seats, Voice Control]","[ABS, Alarm system, Central door lock with rem...","[\nAudi A1: , - 1e eigenaar , - Perfecte staat...",NaN,[\n6\n],NaN
4,Audi A1,Sportback 1.4 TDI S-Tronic S-Line Ext. admired...,Sedans,16790,None,"16,200 km",05/2016,1 previous owner,66 kW,"[, Used, , Diesel (Particulate Filter)]",...,[\n109 g CO2/km (comb)\n],[\nEuro 6\n],"[Air conditioning, Armrest, Automatic climate ...","[Bluetooth, CD player, Hands-free equipment, M...","[Alloy wheels, Sport package, Sport suspension...","[ABS, Central door lock, Driver-side airbag, E...","[\n, Technik & Sicherheit:, Xenon plus, Klimaa...",NaN,NaN,[\nGermany\n]


In [60]:
# Clean strings and commas, then convert to numeric of km
df["km"] = (
    df["km"]
    .astype(str)
    .str.replace("km", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["km"] = pd.to_numeric(df["km"], errors="coerce")

In [61]:
# Clean hp and convert to numeric
df["hp"] = (
    df["hp"]
    .astype(str)
    .str.extract(r"(\d+)")
)

df["hp"] = pd.to_numeric(df["hp"], errors="coerce")

In [62]:
df[["previous_owners","prev_owner"]]

,previous_owners,prev_owner
0,\n2\n,2 previous owners
1,NaN,None
2,\n1\n,1 previous owner
3,\n1\n,1 previous owner
4,\n1\n,1 previous owner
...,...,...
15914,NaN,None
15915,"[\n1\n, \n, 7.4 l/100 km (comb), \n, 9.2 l/100...",1 previous owner
15916,"[\n1\n, \n139 g CO2/km (comb)\n]",1 previous owner
15917,NaN,None


In [63]:
# "previous_owners" and "prev_owner" describe same thing, they will need to merge

df["previous_owners"] = df["previous_owners"].combine_first(df["prev_owner"])

df["previous_owners"] = (
    df["previous_owners"]
    .astype(str)
    .str.extract(r"(\d+)")
)

df["previous_owners"] = pd.to_numeric(df["previous_owners"], errors="coerce")

df = df.drop(columns=["prev_owner"], errors="ignore")

In [64]:
#print(df["previous_owners"])

In [65]:
df.head()

,make_model,short_description,body_type,price,vat,km,registration,hp,type,previous_owners,...,co2_emission,emission_class,comfort_and_convenience,entertainment_and_media,extras,safety_and_security,description,emission_label,gears,country_version
0,Audi A1,Sportback 1.4 TDI S-tronic Xenon Navi Klima,Sedans,15770,VAT deductible,56013.0,01/2016,66.0,"[, Used, , Diesel (Particulate Filter)]",2.0,...,[\n99 g CO2/km (comb)\n],[\nEuro 6\n],"[Air conditioning, Armrest, Automatic climate ...","[Bluetooth, Hands-free equipment, On-board com...","[Alloy wheels, Catalytic Converter, Voice Cont...","[ABS, Central door lock, Daytime running light...","[\n, Sicherheit:, , Deaktivierung für Beifahr...",NaN,NaN,NaN
1,Audi A1,1.8 TFSI sport,Sedans,14500,Price negotiable,80000.0,03/2017,141.0,"[, Used, , Gasoline]",NaN,...,[\n129 g CO2/km (comb)\n],[\nEuro 6\n],"[Air conditioning, Automatic climate control, ...","[Bluetooth, Hands-free equipment, On-board com...","[Alloy wheels, Sport seats, Sport suspension, ...","[ABS, Central door lock, Central door lock wit...",[\nLangstreckenfahrzeug daher die hohe Kilomet...,[\n4 (Green)\n],[\n7\n],NaN
2,Audi A1,Sportback 1.6 TDI S tronic Einparkhilfe plus+m...,Sedans,14640,VAT deductible,83450.0,02/2016,85.0,"[, Used, , Diesel (Particulate Filter)]",1.0,...,[\n99 g CO2/km (comb)\n],[\nEuro 6\n],"[Air conditioning, Cruise control, Electrical ...","[MP3, On-board computer]","[Alloy wheels, Voice Control]","[ABS, Central door lock, Daytime running light...","[\n, Fahrzeug-Nummer: AM-95365, , Ehem. UPE 2...",[\n4 (Green)\n],NaN,NaN
3,Audi A1,1.4 TDi Design S tronic,Sedans,14500,None,73000.0,08/2016,66.0,"[, Used, , Diesel (Particulate Filter)]",1.0,...,[\n99 g CO2/km (comb)\n],[\nEuro 6\n],"[Air suspension, Armrest, Auxiliary heating, E...","[Bluetooth, CD player, Hands-free equipment, M...","[Alloy wheels, Sport seats, Voice Control]","[ABS, Alarm system, Central door lock with rem...","[\nAudi A1: , - 1e eigenaar , - Perfecte staat...",NaN,[\n6\n],NaN
4,Audi A1,Sportback 1.4 TDI S-Tronic S-Line Ext. admired...,Sedans,16790,None,16200.0,05/2016,66.0,"[, Used, , Diesel (Particulate Filter)]",1.0,...,[\n109 g CO2/km (comb)\n],[\nEuro 6\n],"[Air conditioning, Armrest, Automatic climate ...","[Bluetooth, CD player, Hands-free equipment, M...","[Alloy wheels, Sport package, Sport suspension...","[ABS, Central door lock, Driver-side airbag, E...","[\n, Technik & Sicherheit:, Xenon plus, Klimaa...",NaN,NaN,[\nGermany\n]


In [66]:
# Clean registration and create age, because age is more useful than registration

df["registration_year"] = (
    df["registration"]
    .astype(str)
    .str.extract(r"(\d{4})")
)

df["registration_year"] = pd.to_numeric(df["registration_year"], errors="coerce")

df["age"] = 2019 - df["registration_year"]

In [67]:
# Clean firt_registration
df["first_registration"] = (
    df["first_registration"]
    .astype(str)
    .str.extract(r"(\d{4})")
)


In [68]:
# Clean warranty - exract warranty
df["warranty"] = (
    df["warranty"]
    .astype(str)
    .str.extract(r"(\d+)")
)

df["warranty"] = pd.to_numeric(df["warranty"], errors="coerce")

In [69]:
# Clean nr_of_doors
df["nr_of_doors"] = (
    df["nr_of_doors"]
    .astype(str)
    .str.extract(r"(\d+)")
)

df["nr_of_doors"] = pd.to_numeric(df["nr_of_doors"], errors="coerce")


In [70]:
# Clean nr_of_seats

df["nr_of_seats"] = (
    df["nr_of_seats"]
    .astype(str)
    .str.extract(r"(\d+)")
)

df["nr_of_seats"] = pd.to_numeric(df["nr_of_seats"], errors="coerce")

In [71]:
# Clean displacement - remove "cc" and keep engine size.
df["displacement"] = (
    df["displacement"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.extract(r"(\d+)")
)

df["displacement"] = pd.to_numeric(df["displacement"], errors="coerce")

In [72]:
# Clean cylinders
df["cylinders"] = (
    df["cylinders"]
    .astype(str)
    .str.extract(r"(\d+)")
)

df["cylinders"] = pd.to_numeric(df["cylinders"], errors="coerce")

In [73]:
# Clean weight
df["weight"] = (
    df["weight"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.extract(r"(\d+)")
)

df["weight"] = pd.to_numeric(df["weight"], errors="coerce")

In [74]:
# Clean gears
df["gears"] = (
    df["gears"]
    .astype(str)
    .str.extract(r"(\d+)")
)

df["gears"] = pd.to_numeric(df["gears"], errors="coerce")

In [75]:
# upholstery_type - upholstery has mixed color/material information.
df["upholstery_type"] = df["upholstery"].astype(str).str.extract(
    r"(Cloth|Leather|Part leather|Full leather|Velour|Alcantara)",
    flags=re.IGNORECASE
)

df["upholstery_type"] = df["upholstery_type"].str.lower()

In [76]:
# Clean paint_type
df["paint_type"] = (
    df["paint_type"]
    .astype(str)
    .str.replace("\n", "", regex=False)
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.strip()
)

df["paint_type"] = df["paint_type"].replace("nan", np.nan)

In [77]:
#df.head()

In [78]:
# keep for get_dummies later
dummy_cols = [
    "comfort_and_convenience",
    "entertainment_and_media",
    "extras",
    "safety_and_security"
]

In [80]:
duplicate_check_cols = [
    "make_model",
    "short_description",
    "body_type",
    "price",
    "km",
    "registration",
    "hp",
    "gearing_type",
    "body_color"
]

# remove columns that contain lists
safe_duplicate_cols = [
    col for col in duplicate_check_cols
    if col in df.columns and not df[col].apply(lambda x: isinstance(x, list)).any()
]

df = df.drop_duplicates(subset=safe_duplicate_cols)

df.shape

(14101, 45)

In [ ]:
#df.info()
#df.shape
#df.isnull().sum().sort_values(ascending=False).head(20)

In [81]:
df.to_csv("../data/processed/clean_data.csv", index=False)